# Day10 · E0 zero-shot · Văn Thành Huy
Không training. Nạp Qwen gốc, cùng quantization/prompt/256 output tokens như E1; kiểm tra val rồi chấm đủ 311 ảnh test.
Kaggle: Internet ON, T4 GPU, Add Input dataset road-damage. Chạy tuần tự và tải day10_results.zip.
Không nạp adapter E1. Không điều chỉnh prompt theo test. Dự đoán cần GPU, chưa đo thời gian toàn bộ E0.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, zipfile
REPO_URL = "https://github.com/thanhhuyvan/K4-Track-4-Day10-VanThanhHuy-2A202602763--VisionLanguageModel.git"
ROOT = Path("/kaggle/working/day10")
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
# Có thể trỏ đến repo được upload dưới dạng Kaggle Input nếu GitHub chưa có mã đề.
REPO_INPUT = None
DATA_INPUT = None  # None: tìm tự động trong /kaggle/input; hoặc nhập đường dẫn dataset
LOAD_MODEL = False  # đổi True khi đã bật GPU và sẵn sàng tải model

def run(*args):
    subprocess.run([str(x) for x in args], check=True, cwd=ROOT if ROOT.exists() else None)

if not ROOT.exists():
    if REPO_INPUT:
        shutil.copytree(REPO_INPUT, ROOT)
    else:
        run("git", "clone", REPO_URL, ROOT)
for required in ["scripts/prepare_data.py", "scripts/score.py", "requirements-colab.txt"]:
    assert (ROOT / required).is_file(), f"Repo thiếu {required}; upload repo vào Kaggle Input và đặt REPO_INPUT."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
(ROOT / "work").mkdir(exist_ok=True)
(ROOT / "report/evidence").mkdir(parents=True, exist_ok=True)
# Ghi hash để phát hiện vô tình sửa công cụ đề.
PROTECTED = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
             for folder in ["scripts", "tests"] for p in (ROOT / folder).rglob("*.py")}
print("Repo:", ROOT)


## 1. Môi trường — loại torchao cũ không dùng trong bài

In [ ]:
run(sys.executable, "-m", "pip", "uninstall", "-y", "torchao")
run(sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "pytest", "matplotlib")
check = subprocess.run([sys.executable, "-m", "pip", "check"], cwd=ROOT, capture_output=True, text=True)
print(check.stdout, check.stderr)
# pip check có thể báo xung đột thư viện khác trong image Kaggle; không giấu output.
run(sys.executable, "-m", "pytest", "-q")


In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes
runtime = {"python": sys.version, "torch": torch.__version__,
           "transformers": transformers.__version__, "peft": peft.__version__,
           "accelerate": accelerate.__version__, "bitsandbytes": bitsandbytes.__version__,
           "cuda_available": torch.cuda.is_available()}
if torch.cuda.is_available():
    runtime["gpus"] = [{"name": torch.cuda.get_device_name(i),
                        "vram_gb": torch.cuda.get_device_properties(i).total_memory / 2**30}
                       for i in range(torch.cuda.device_count())]
print(json.dumps(runtime, indent=2))
(ROOT / "work/runtime.json").write_text(json.dumps(runtime, indent=2))


## 2. Dữ liệu và kiểm tra không trùng phiên quay

In [ ]:
from scripts.prepare_data import _find_root
base = Path(DATA_INPUT) if DATA_INPUT else Path("/kaggle/input")
try:
    source = _find_root(base)
except FileNotFoundError:
    raise RuntimeError("Chưa tìm thấy images/ và labels-YOLO/. Add Input dataset hoặc đặt DATA_INPUT đúng thư mục.")
run(sys.executable, "scripts/prepare_data.py", "--source", source)
meta = json.loads((ROOT / "data/splits/meta.json").read_text())
splits = {s: [json.loads(line) for line in (ROOT / f"data/splits/{s}.jsonl").read_text().splitlines() if line.strip()]
          for s in ["train", "val", "test"]}
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not sessions["train"] & sessions["val"]
assert not sessions["train"] & sessions["test"]
assert not sessions["val"] & sessions["test"]
print(json.dumps(meta["counts"], indent=2))
print("dataset_version:", meta["dataset_version"], "— không trùng phiên quay")


## 3. Nhãn — ghi log box lỗi, giữ nguyên dataset và công cụ chấm

In [ ]:
import random, time, math, gc, re
from torch.utils.data import DataLoader
from PIL import Image

def target_box(obj):
    xc,yc,w,h = map(float,obj["yolo"])
    if w <= 0 or h <= 0: return None, "skip_zero_area"
    floats = [max(0.,min(1000.,v*1000)) for v in [xc-w/2,yc-h/2,xc+w/2,yc+h/2]]
    if floats[2]<=floats[0] or floats[3]<=floats[1]: return None,"skip_after_clipping"
    x1,y1,x2,y2=map(round,floats)
    adjusted=x2<=x1 or y2<=y1
    if x2<=x1: x1=min(x1,999); x2=x1+1
    if y2<=y1: y1=min(y1,999); y2=y1+1
    return {"label":obj["label"],"bbox_2d":[x1,y1,x2,y2]}, "expand_rounding_to_one_unit" if adjusted else None

def objects_to_answer(objects):
    boxes=[b for obj in objects for b,_ in [target_box(obj)] if b is not None]
    boxes.sort(key=lambda b:(b["bbox_2d"][1],b["bbox_2d"][0]))
    return json.dumps(boxes,separators=(",",":"))

audit=[]
for row in splits["train"]:
    for obj in row["objects"]:
        _,action=target_box(obj)
        if action: audit.append({"image_id":row["image_id"],"object":obj,"action":action})
(ROOT/"work/train_label_audit.json").write_text(json.dumps(audit,indent=2))
print("Label audit:",audit)
PROMPT='Locate all pothole, crack and manhole objects. Return only a JSON list of objects with label and bbox_2d=[x1,y1,x2,y2]. Use integer coordinates normalized to 0–1000 relative to image width and height. Return [] if none.'


## 4. Model gốc — chỉ lượng tử hóa LLM, xác nhận ViT và lm_head

In [ ]:
from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
from accelerate import init_empty_weights
from peft import LoraConfig, get_peft_model, PeftModel
import bitsandbytes as bnb
assert torch.cuda.is_available(), "Bật GPU trước khi tiếp tục"
processor=AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side="left"

def release_model():
    global model
    if "model" in globals(): del model
    gc.collect(); torch.cuda.empty_cache()

def load_base():
    config=AutoConfig.from_pretrained(MODEL_ID)
    with init_empty_weights(): skeleton=Qwen3VLForConditionalGeneration(config)
    skip=[name for name,module in skeleton.named_modules()
          if isinstance(module,torch.nn.Linear) and ("visual" in name.split(".") or name.split(".")[-1]=="lm_head")]
    assert skip
    del skeleton
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True,llm_int8_skip_modules=skip)
    base=Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID,
        quantization_config=quant,dtype=torch.float16,device_map={"":0})
    bad=[name for name,module in base.named_modules()
         if ("visual" in name.split(".") or name.split(".")[-1]=="lm_head")
         and isinstance(module,(bnb.nn.Linear4bit,bnb.nn.Linear8bitLt))]
    assert not bad,f"ViT/lm_head bị quantize: {bad[:10]}"
    return base

def setup_e1(base):
    for p in base.parameters(): p.requires_grad=False
    pattern=r".*language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
    matched=[name for name,_ in base.named_modules() if re.fullmatch(pattern,name)]
    assert len(matched)==252, f"LLM targets: {len(matched)} thay vì 252"
    base.enable_input_require_grads()
    trained=get_peft_model(base,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,
        target_modules=pattern,task_type="CAUSAL_LM"))
    for name,p in trained.named_parameters():
        if p.requires_grad:
            assert "lora_" in name and "language_model" in name, name
            p.data=p.data.float()
    trained.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
    trained.config.use_cache=False
    trained.print_trainable_parameters()
    return trained


## 5. Model gốc và preview val

In [ ]:
release_model()
model=load_base()
model.eval();model.config.use_cache=True
processor.tokenizer.padding_side="left"

def predict_e0(split="val",limit=None):
    if split=="test":assert limit is None
    rows=splits[split] if limit is None else splits[split][:limit]
    path=ROOT/"work"/f"e0_{split}.jsonl"; start=time.perf_counter()
    with path.open("w",encoding="utf-8") as f:
        for i,row in enumerate(rows,1):
            with Image.open(ROOT/"data"/row["image_path"]) as im:image=im.convert("RGB")
            messages=[{"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT}]}]
            inputs=processor.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
                return_dict=True,return_tensors="pt").to("cuda:0")
            with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
                generated=model.generate(**inputs,max_new_tokens=256,do_sample=False)
            raw=processor.batch_decode(generated[:,inputs["input_ids"].shape[1]:],skip_special_tokens=True)[0]
            f.write(json.dumps({"image_id":row["image_id"],"raw":raw},ensure_ascii=False)+"\n");f.flush()
            if i<=3:print(row["image_id"],raw)
            if i%25==0 or i==len(rows):print(f"{split}: {i}/{len(rows)}, {(time.perf_counter()-start)/60:.1f} phút")
            del inputs,generated
    return path
preview=predict_e0("val",limit=5)
from scripts.score import parse_output
preview_rows=[json.loads(line) for line in preview.read_text().splitlines()]
print("Val parse hợp lệ:",sum(parse_output(r["raw"]) is not None for r in preview_rows),"/",len(preview_rows))


## 6. Chấm test E0

In [ ]:
pred=predict_e0("test")
rows=[json.loads(line) for line in pred.read_text().splitlines()]
assert len(rows)==len(splits["test"])
assert {r["image_id"] for r in rows}=={r["image_id"] for r in splits["test"]}
run(sys.executable,"scripts/score.py","--pred",str(pred),"--split","test","--name","E0","--out","report/evidence/e0.json")
print(json.loads((ROOT/"report/evidence/e0.json").read_text()))


## 7. Xuất kết quả về local

In [ ]:
from IPython.display import FileLink,display
for rel,expected in PROTECTED.items():
    assert hashlib.sha256((ROOT/rel).read_bytes()).hexdigest()==expected,f"Đã sửa công cụ đề: {rel}"
archive=Path("/kaggle/working/day10_results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [ROOT/"report",ROOT/"work"]:
        for p in folder.rglob("*"):
            rel=p.relative_to(ROOT)
            if p.is_file() and "checkpoints" not in rel.parts and p.suffix.lower() in {".json",".jsonl",".png",".csv",".txt",".log"}:
                z.write(p,str(rel))
    z.write(ROOT/"data/splits/meta.json","data/splits/meta.json")
print("Kết quả:",archive)
display(FileLink(str(archive)))
